# Task 08 -- Masked-HOOI convergence sensitivity (`docs/open_questions.md` §5.4)

Multi-initialization sensitivity check for masked HOOI's imputation of
land/ice cells, per `tucker_decomposition.md` §6's proposed mitigation.
See `tasks/08-masked-hooi-convergence.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import tensorly as tl
import xarray as xr

from core.config import canonical_dims
from train.ingest import load_training_window
from train.masking import height_log_energy

tl.set_backend("numpy")

SEED = 42
DATA_DIR = Path("../data")
GAUSS_DIR = Path("../results/task07/gaussianize")

# Redefined 2026-09-29 (task 07, tasks/08-masked-hooi-convergence.md "Update"):
# era-2 grid (621,1440), reusing task 07's already-fitted transformers and
# already-computed ocean mask instead of a from-scratch era-3 raw-`wsh` load
# with a mask derived from `wsh`'s own (bug-conflated, task 17) NaN pattern.
# A short 2-month sub-window keeps the N_INITS=8 sweep below tractable --
# the fitted climatology/standardize transformers generalize to any
# sub-window of the span they were fit over, so this is still a faithful
# application of the real production pipeline, just over less time.
T_START = datetime(2024, 1, 1, tzinfo=timezone.utc)
T_END = datetime(2024, 3, 1, tzinfo=timezone.utc)
VARIABLE = "wsh"  # large contiguous land regions in the wave grid make this a good stress test

ocean_mask = np.load(GAUSS_DIR / "era2_ocean_mask.npy")  # (621,1440), True=ocean
clim = joblib.load(GAUSS_DIR / f"{VARIABLE}_climatology.joblib")
std = joblib.load(GAUSS_DIR / f"{VARIABLE}_standardize.joblib")

ds = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
raw = ds[VARIABLE].transpose(*canonical_dims("wave")).values.astype(np.float32)
time_coord = ds["time"].values

# Same ocean_only_sentinel_fill pipeline as notebooks/07_placement2_tucker_on_anomalies.py
# (duplicated rather than imported, matching this task's small-script-helper
# convention): ocean NaN -> sentinel 0.0 (VALID), land NaN left untouched (INVALID).
nan_mask = np.isnan(raw)
ocean_3d = np.broadcast_to(ocean_mask[None, :, :], raw.shape)
filled = np.where(nan_mask & ocean_3d, 0.0, raw)
tucker_mask = ocean_3d.astype(np.int8)
x = height_log_energy(filled, floor_m=0.01)

lat_dim, lon_dim = canonical_dims("wave")[1], canonical_dims("wave")[2]
ds_x = xr.Dataset({VARIABLE: (("time", lat_dim, lon_dim), x)}, coords={"time": time_coord})
anomalies = clim.transform(ds_x)  # already-fitted -- transform only, no re-fit
standardized = std.transform(anomalies)
z_full = standardized[VARIABLE].values.astype(np.float32)
z_full = np.where(np.isnan(z_full), 0.0, z_full)
mask_int8 = tucker_mask

print(f"{VARIABLE}: shape {z_full.shape}, {ocean_mask.mean():.1%} ocean")


## Multiple random initializations of masked HOOI, same rank/data

In [ ]:
from tensorly.tenalg import multi_mode_dot

from train.masked_tucker import partial_tucker_lowmem

# Small rank (~3% of the era-2 grid, well below any rank task 07 tested) and a
# fixed n_iter_max (not full convergence) -- deliberate choices for local
# tractability across 8 separate fits: this task's question is spread across
# inits, not full convergence, so early divergence across seeds is itself
# the signal. Sanity-check one init's wall time before running all 8
# unattended (see tasks/08-masked-hooi-convergence.md's "Update" note).
SPATIAL_MODES = [1, 2]  # (time, lat, lon) -- time (mode 0) stays full-rank
R_LAT, R_LON = 20, 40
N_INITS = 8
N_ITER_MAX = 100

# partial_tucker_lowmem only supports init="svd" or an (core, factors) tuple
# (init="random" is unsupported -- "no call site here ever uses it", per its
# own docstring). Build a genuinely random init tuple per seed here instead,
# following tensorly's own random-factor convention (uniform, not SVD-anchored)
# but projecting through multi_mode_dot for a correctly-shaped initial core
# (tensorly's own `initialize_tucker` "random" branch doesn't preserve the
# untouched time axis, which this repo's ALS loop needs).
def _random_tucker_init(tensor, rank, modes, seed):
    rng = np.random.default_rng(seed)
    factors = [rng.random(size=(tensor.shape[mode], r)) for mode, r in zip(modes, rank)]
    core = multi_mode_dot(tensor, factors, modes=modes, transpose=True)
    return core, factors


reconstruction_errors = []
imputed_land_stats = []
for seed in range(N_INITS):
    init = _random_tucker_init(z_full, [R_LAT, R_LON], SPATIAL_MODES, seed)
    (core, factors), _ = partial_tucker_lowmem(
        z_full, modes=SPATIAL_MODES, rank=[R_LAT, R_LON], init=init,
        tol=1e-5, n_iter_max=N_ITER_MAX, random_state=seed, mask=mask_int8,
        svd="randomized_svd", svd_mask_repeats=0,
    )
    recon = multi_mode_dot(core, factors, modes=SPATIAL_MODES)
    err = float(
        np.linalg.norm((recon - z_full) * mask_int8) / np.linalg.norm(z_full * mask_int8)
    )
    reconstruction_errors.append(err)
    imputed_land_stats.append(float(recon[:, ~ocean_mask].mean()))
    print(f"init {seed}: relative error {err:.5f}, mean imputed-land value {imputed_land_stats[-1]:.4f}")

reconstruction_errors = np.array(reconstruction_errors)
imputed_land_stats = np.array(imputed_land_stats)
print(f"\nerror spread: mean {reconstruction_errors.mean():.5f}, std {reconstruction_errors.std():.5f} "
      f"({reconstruction_errors.std() / reconstruction_errors.mean():.1%} relative)")
print(f"imputed-land spread: mean {imputed_land_stats.mean():.4f}, std {imputed_land_stats.std():.4f}")


## Decision-rule diagnostic

Per `tucker_decomposition.md` §6: flag as "initialization-sensitive" if
either spread above has high relative std. Repeat for a variable with a
smaller contiguous land region as a control (e.g. `sigwh` at a coastal-
heavy sub-region) to see whether sensitivity correlates with land-region
size, as hypothesized. Record findings in `tasks/08-masked-hooi-convergence.md` §4.